##### Copyright 2024 The TensorFlow Authors.

In [ ]:
#@title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

#Wprowadzenie do Autoenkoderów
Niniejszy notatnik jest adaptacją oficjalnego samouczka TensorFlow Authors (Copyright 2024). Został on zmodyfikowany na potrzeby zajęć laboratoryjnych, aby zilustrować przejście od obliczeń ręcznych w arkuszu kalkulacyjnym do automatyzacji procesów diagnostycznych w środowisku Python.

Ten notatnik przedstawia trzy przykłady zastosowania autoenkoderów: podstawy, usuwanie szumu z obrazów (denoising) oraz wykrywanie anomalii.

Autoenkoder to specjalny typ sieci neuronowej, który jest trenowany tak, aby kopiował wejście na wyjście. Na przykład, mając obraz odręcznie napisanej cyfry, autoenkoder najpierw kompresuje obraz do reprezentacji w niższej przestrzeni wymiarowej (latent-space), a następnie dekoduje tę reprezentację z powrotem do obrazu. Autoenkoder uczy się kompresować dane, jednocześnie minimalizując błąd rekonstrukcji.

## Import bibliotek

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf

from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from tensorflow.keras import layers, losses
from tensorflow.keras.datasets import fashion_mnist
from tensorflow.keras.models import Model

#Przykład 1: Podstawowy Autoenkoder


## Załadowanie zbioru danych
Na początek wytrenujesz podstawowy autoenkoder, korzystając ze zbioru danych Fashion MNIST. Każdy obraz w tym zbiorze danych ma rozmiar 28x28 pikseli.

In [ ]:
(x_train, _), (x_test, _) = fashion_mnist.load_data()

x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

print (x_train.shape)
print (x_test.shape)

#Definicja modelu
Zdefiniujemy autoenkoder z dwiema warstwami: encoder, który kompresuje dane do 64-wymiarowego wektora, oraz decoder, który rekonstruuje obraz.



In [ ]:
class Autoencoder(Model):
  def __init__(self, latent_dim, shape):
    super(Autoencoder, self).__init__()
    self.latent_dim = latent_dim
    self.shape = shape
    self.encoder = tf.keras.Sequential([
      layers.Flatten(),
      layers.Dense(latent_dim, activation='relu'),
    ])
    self.decoder = tf.keras.Sequential([
      layers.Dense(int(tf.math.reduce_prod(shape).numpy()), activation='sigmoid'), # Cast to int
      layers.Reshape(shape)
    ])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded


shape = x_test.shape[1:]
latent_dim = 64
autoencoder = Autoencoder(latent_dim, shape)


#Kompilacja i trening modelu
W tym miejscu definiujemy, jak model ma mierzyć błędy (MeanSquaredError) i jak ma je naprawiać (adam).

In [ ]:
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

In [ ]:
autoencoder.fit(x_train, x_train,
                epochs=10,
                shuffle=True,
                validation_data=(x_test, x_test))

#Wyświetlanie wyników (Rekonstrukcja)
Po wytrenowaniu modelu chcemy sprawdzić, jak dobrze "odrysowuje" on ubrania po ich wcześniejszym skompresowaniu do 64 liczb.

In [ ]:
encoded_imgs = autoencoder.encoder(x_test).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):
  # display original
  ax = plt.subplot(2, n, i + 1)
  plt.imshow(x_test[i])
  plt.title("original")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)

  # display reconstruction
  ax = plt.subplot(2, n, i + 1 + n)
  plt.imshow(decoded_imgs[i])
  plt.title("reconstructed")
  plt.gray()
  ax.get_xaxis().set_visible(False)
  ax.get_yaxis().set_visible(False)
plt.show()

##Przykład 2: Usuwanie szumu (Denoising)
Autoenkoder można również wytrenować tak, aby usuwał szum z obrazów. Wytrenujemy model na zaszumionych obrazach i nauczymy go odtwarzać oryginał.



In [ ]:
(x_train, _), (x_test, _) = fashion_mnist.load_data()

In [ ]:
x_train = x_train.astype('float32') / 255.
x_test = x_test.astype('float32') / 255.

x_train = x_train[..., tf.newaxis]
x_test = x_test[..., tf.newaxis]

print(x_train.shape)

Przygotowanie zaszumionych danych

In [ ]:
noise_factor = 0.2
x_train_noisy = x_train + noise_factor * tf.random.normal(shape=x_train.shape)
x_test_noisy = x_test + noise_factor * tf.random.normal(shape=x_test.shape)

x_train_noisy = tf.clip_by_value(x_train_noisy, clip_value_min=0., clip_value_max=1.)
x_test_noisy = tf.clip_by_value(x_test_noisy, clip_value_min=0., clip_value_max=1.)

In [ ]:
n = 10
plt.figure(figsize=(20, 2))
for i in range(n):
    ax = plt.subplot(1, n, i + 1)
    plt.title("original + noise")
    plt.imshow(tf.squeeze(x_test_noisy[i]))
    plt.gray()
plt.show()

W tym przykładzie wytrenujesz autoenkoder splotowy, używając warstw [Conv2D](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2D) w enkoderze i warstw [Conv2DTranspose](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Conv2DTranspose) w dekoderze.

In [ ]:
class Denoise(Model):
  def __init__(self):
    super(Denoise, self).__init__()
    self.encoder = tf.keras.Sequential([
      layers.Input(shape=(28, 28, 1)),
      layers.Conv2D(16, (3, 3), activation='relu', padding='same', strides=2),
      layers.Conv2D(8, (3, 3), activation='relu', padding='same', strides=2)])

    self.decoder = tf.keras.Sequential([
      layers.Conv2DTranspose(8, kernel_size=3, strides=2, activation='relu', padding='same'),
      layers.Conv2DTranspose(16, kernel_size=3, strides=2, activation='relu', padding='same'),
      layers.Conv2D(1, kernel_size=(3, 3), activation='sigmoid', padding='same')])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

autoencoder = Denoise()

In [ ]:
autoencoder.compile(optimizer='adam', loss=losses.MeanSquaredError())

In [ ]:
autoencoder.fit(x_train_noisy, x_train,
                epochs=10,
                shuffle=True,
                validation_data=(x_test_noisy, x_test))

Przyjrzyjmy się podsumowaniu działania enkodera. Zwróć uwagę, jak obrazy są zmniejszane z rozmiaru 28x28 do 7x7.

In [ ]:
autoencoder.encoder.summary()

Dekoder przekształca obrazy z powrotem z formatu 7x7 do 28x28.

In [ ]:
autoencoder.decoder.summary()

Wykreślanie zarówno obrazów zaszumionych, jak i obrazów bez szumów wytworzonych przez autoenkoder.

In [ ]:
encoded_imgs = autoencoder.encoder(x_test_noisy).numpy()
decoded_imgs = autoencoder.decoder(encoded_imgs).numpy()

In [ ]:
n = 10
plt.figure(figsize=(20, 4))
for i in range(n):

    # display original + noise
    ax = plt.subplot(2, n, i + 1)
    plt.title("original + noise")
    plt.imshow(tf.squeeze(x_test_noisy[i]))
    plt.gray()
    ax.get_xaxis().set_visible(False)
    ax.get_yaxis().set_visible(False)

    # display reconstruction
    bx = plt.subplot(2, n, i + n + 1)
    plt.title("reconstructed")
    plt.imshow(tf.squeeze(decoded_imgs[i]))
    plt.gray()
    bx.get_xaxis().set_visible(False)
    bx.get_yaxis().set_visible(False)
plt.show()

### Przykład 3: Wykrywanie anomalii
W tym przykładzie wytrenujemy autoenkoder do wykrywania anomalii w zbiorze danych ECG5000 (elektrokardiogramy). Zbiór zawiera 5000 zapisów EKG, z których każdy ma 140 punktów danych.
Użyjesz uproszczonej wersji zbioru danych, w którym każdy przykład został oznaczony etykietą 0 (odpowiadającą rytmowi nieprawidłowemu) lub 1 (odpowiadającej rytmowi normalnemu). Interesuje Cię identyfikacja rytmów nieprawidłowych.

Uwaga: Jest to zbiór danych z etykietami, więc można go zdefiniować jako problem uczenia nadzorowanego. Celem tego przykładu jest zilustrowanie koncepcji wykrywania anomalii, które można zastosować do większych zbiorów danych, w których nie ma dostępnych etykiet (na przykład, jeśli masz wiele tysięcy rytmów prawidłowych i tylko niewielką liczbę rytmów nieprawidłowych).

Jak wykryjesz anomalie za pomocą autoenkodera? Przypomnij sobie, że autoenkoder jest trenowany w celu minimalizacji błędów rekonstrukcji. Wytrenujesz autoenkoder tylko na rytmach prawidłowych, a następnie użyjesz go do rekonstrukcji wszystkich danych. Nasza hipoteza jest taka, że ​​rytmy nieprawidłowe będą miały wyższe błędy rekonstrukcji. Następnie zaklasyfikujesz rytm jako anomalię, jeśli błąd rekonstrukcji przekroczy ustalony próg.


### Przygotowanie danych EKG

Zbiór pochodzi z: [timeseriesclassification.com](http://www.timeseriesclassification.com/description.php?Dataset=ECG5000).


In [ ]:
dataframe = pd.read_csv('http://storage.googleapis.com/download.tensorflow.org/data/ecg.csv', header=None)
raw_data = dataframe.values
dataframe.head()

#Trenowanie modelu
Będziemy trenować autoenkoder wyłącznie na danych oznaczonych jako 1 (normalne).

In [ ]:
# Ostatnia kolumna to etykieta (1 - normalne, 0 - anomalia)
labels = raw_data[:, -1]


data = raw_data[:, 0:-1]

train_data, test_data, train_labels, test_labels = train_test_split(
    data, labels, test_size=0.2, random_state=21
)

Normalizacja danych do: `[0,1]`.


In [ ]:
min_val = tf.reduce_min(train_data)
max_val = tf.reduce_max(train_data)

train_data = (train_data - min_val) / (max_val - min_val)
test_data = (test_data - min_val) / (max_val - min_val)

train_data = tf.cast(train_data, tf.float32)
test_data = tf.cast(test_data, tf.float32)

In [ ]:
train_labels = train_labels.astype(bool)
test_labels = test_labels.astype(bool)

normal_train_data = train_data[train_labels]
normal_test_data = test_data[test_labels]

anomalous_train_data = train_data[~train_labels]
anomalous_test_data = test_data[~test_labels]

Plot a normal ECG.

In [ ]:
plt.grid()
plt.plot(np.arange(140), normal_train_data[0])
plt.title("A Normal ECG")
plt.show()

Plot an anomalous ECG.

In [ ]:
plt.grid()
plt.plot(np.arange(140), anomalous_train_data[0])
plt.title("An Anomalous ECG")
plt.show()

### Build the model

In [ ]:
class AnomalyDetector(Model):
  def __init__(self):
    super(AnomalyDetector, self).__init__()
    self.encoder = tf.keras.Sequential([
      layers.Dense(32, activation="relu"),
      layers.Dense(16, activation="relu"),
      layers.Dense(8, activation="relu")])

    self.decoder = tf.keras.Sequential([
      layers.Dense(16, activation="relu"),
      layers.Dense(32, activation="relu"),
      layers.Dense(140, activation="sigmoid")])

  def call(self, x):
    encoded = self.encoder(x)
    decoded = self.decoder(encoded)
    return decoded

autoencoder = AnomalyDetector()

In [ ]:
autoencoder.compile(optimizer='adam', loss='mae')

Należy zauważyć, że autoenkoder jest trenowany wyłącznie przy użyciu normalnych EKG, ale oceniany przy użyciu pełnego zestawu testowego.

In [ ]:
history = autoencoder.fit(normal_train_data, normal_train_data,
          epochs=20,
          batch_size=512,
          validation_data=(test_data, test_data),
          shuffle=True)

In [ ]:
plt.plot(history.history["loss"], label="Training Loss")
plt.plot(history.history["val_loss"], label="Validation Loss")
plt.legend()

EKG zostanie sklasyfikowane jako anomalne, jeśli błąd rekonstrukcji jest większy niż jedno odchylenie standardowe od normalnego zapisu treningowego. Najpierw narysujmy wykres normalnego EKG z zestawu treningowego, rekonstrukcji po zakodowaniu i zdekodowaniu przez autoenkoder oraz błędu rekonstrukcji.

In [ ]:
encoded_data = autoencoder.encoder(normal_test_data).numpy()
decoded_data = autoencoder.decoder(encoded_data).numpy()

plt.plot(normal_test_data[0], 'b')
plt.plot(decoded_data[0], 'r')
plt.fill_between(np.arange(140), decoded_data[0], normal_test_data[0], color='lightcoral')
plt.legend(labels=["Input", "Reconstruction", "Error"])
plt.show()

Utwórz podobny wykres, tym razem dla przykładowego testu anomalii.

In [ ]:
encoded_data = autoencoder.encoder(anomalous_test_data).numpy()
decoded_data = autoencoder.decoder(encoded_data).numpy()

plt.plot(anomalous_test_data[0], 'b')
plt.plot(decoded_data[0], 'r')
plt.fill_between(np.arange(140), decoded_data[0], anomalous_test_data[0], color='lightcoral')
plt.legend(labels=["Input", "Reconstruction", "Error"])
plt.show()

##Histogram błędów
Przygotujemy wizualizację rozkładu błędów.

In [ ]:
reconstructions = autoencoder.predict(normal_train_data)
train_loss = tf.keras.losses.mae(reconstructions, normal_train_data)

plt.hist(train_loss[None,:], bins=50)
plt.xlabel("Wartość błędu")
plt.ylabel("Liczba przypadków")
plt.show()

#Ustalenie progu wykrywania (Threshold)
Po wytrenowaniu modelu musimy ustalić od jakiego progu próbkę uważamy za anomalię. Możnaby odczytać ją "na oko" - tam gdzie kończy się górka. My zrobimy to zgodnie ze statystyką.
Ustalimy próg wykrywania anomalii jako jedno odchylenie standardowe powyżej średniego błędu dla normalnych danych.
Czy na oko też tak byśmy wskazali?


In [ ]:
threshold = np.mean(train_loss) + np.std(train_loss)
print("Threshold: ", threshold)

Jeśli przeanalizujesz błąd rekonstrukcji dla nietypowych przykładów w zbiorze testowym, zauważysz, że większość z nich ma większy błąd rekonstrukcji niż próg. Zmieniając próg, możesz dostosować [precyzję](https://developers.google.com/machine-learning/glossary#precision) i [odwołanie](https://developers.google.com/machine-learning/glossary#recall) swojego klasyfikatora.

In [ ]:
reconstructions = autoencoder.predict(anomalous_test_data)
test_loss = tf.keras.losses.mae(reconstructions, anomalous_test_data)

plt.hist(test_loss[None, :], bins=50)
plt.xlabel("Wartość błędu dla anomalii")
plt.ylabel("Ilość przypadków")
plt.show()

W inżynierii maszynowej dążymy do tego, aby te dwa histogramy (dla danych zdrowych i anomalii) były od siebie jak najbardziej odseparowane.

Histogram 1 (Train loss/Normal): Powinien być skupiony blisko zera (lewa strona osi X).

Histogram 2 (Test loss/Anomalous): Powinien być przesunięty mocno w prawo (wysokie wartości błędu).

##Metryki klasyfikacji:
Accuracy (Dokładność):
*   Pytanie: „Ile razy ogółem model miał rację?”
*   Przykład: Jeśli na 100 maszyn model poprawnie rozpoznał stan 95 z nich, Accuracy wynosi 95%.

Precision (Precyzja):
*   Pytanie: „Jeśli model mówi, że serce jest ZDROWE, to jaka jest szansa, że ono naprawdę jest zdrowe?”
*   Zastosowanie: Ważne, żeby nie uspokajać pacjenta (lub operatora maszyny), gdy w rzeczywistości dzieje się coś złego.

Recall (Czułość):
*   Pytanie: „Ile ze wszystkich faktycznie chorych przypadków udało nam się poprawnie wyłapać?”
*   Zastosowanie: Pokazuje, czy nasz model nie jest zbyt "podejrzliwy" i nie wysyła zdrowych ludzi na niepotrzebne leczenie.






In [ ]:
def predict(model, data, threshold):
  reconstructions = model(data)
  loss = tf.keras.losses.mae(reconstructions, data)
  return tf.math.less(loss, threshold)

def print_stats(predictions, labels):
  print("Accuracy = {}".format(accuracy_score(labels, predictions)))
  print("Precision = {}".format(precision_score(labels, predictions)))
  print("Recall = {}".format(recall_score(labels, predictions)))

In [ ]:
preds = predict(autoencoder, test_data, threshold)
print_stats(preds, test_labels)

##Zadanie 1:
W zadaniu z odszumianiem obrazów: dla trzech różnych poziomów szumu (np. 0.1, 0.3, 0.5) wygeneruj zestawienie wizualne kilku przykładowych obrazów i oblicz dla nich metrykę SSIM (Structural Similarity Index). Czy możesz wskazać, przy którym poziomie szumu system przestaje poprawnie rekonstruować znaki alfanumeryczne?
##Zadanie 2:
Dla przykładu z EKG porównaj macierze pomyłek dla dwóch różnych poziomów progu wykrywalności - jednego statystycznego i drugiego ustalonego metodą wizualną.

## Next steps

To learn more about anomaly detection with autoencoders, check out this excellent [interactive example](https://anomagram.fastforwardlabs.com/#/) built with TensorFlow.js by Victor Dibia. For a real-world use case, you can learn how [Airbus Detects Anomalies in ISS Telemetry Data](https://blog.tensorflow.org/2020/04/how-airbus-detects-anomalies-iss-telemetry-data-tfx.html) using TensorFlow. To learn more about the basics, consider reading this [blog post](https://blog.keras.io/building-autoencoders-in-keras.html) by François Chollet. For more details, check out chapter 14 from [Deep Learning](https://www.deeplearningbook.org/) by Ian Goodfellow, Yoshua Bengio, and Aaron Courville.


In [ ]:
# Naprawa: Upewnij się, że używamy instancji modelu Denoise, a nie AnomalyDetector
# Jeśli zmienna 'autoencoder' została nadpisana przez EKG, musimy stworzyć nową instancję Denoise
denoise_model = Denoise()
denoise_model.compile(optimizer='adam', loss='mse')
# Trenujemy krótko, aby umożliwić demonstrację (lub używamy wag jeśli były zapisane)
denoise_model.fit(x_train_noisy, x_train, epochs=1, batch_size=256, verbose=0)

noise_levels = [0.1, 0.3, 0.5]
n_images = 5
sample_images = x_test[:n_images]

for level in noise_levels:
    noisy_samples = sample_images + level * tf.random.normal(shape=sample_images.shape)
    noisy_samples = tf.clip_by_value(noisy_samples, clip_value_min=0., clip_value_max=1.)

    # Używamy poprawnej instancji modelu
    reconstructions = denoise_model(noisy_samples)

    ssim_values = tf.image.ssim(sample_images, reconstructions, max_val=1.0)
    avg_ssim = tf.reduce_mean(ssim_values)

    plt.figure(figsize=(15, 6))
    for i in range(n_images):
        ax = plt.subplot(2, n_images, i + 1)
        plt.title(f"Noisy ({level})")
        plt.imshow(tf.squeeze(noisy_samples[i]))
        plt.gray()
        ax.axis('off')

        bx = plt.subplot(2, n_images, i + n_images + 1)
        plt.title(f"Reconstructed")
        plt.imshow(tf.squeeze(reconstructions[i]))
        plt.gray()
        bx.axis('off')

    plt.suptitle(f"Noise Level: {level} | Average SSIM: {avg_ssim:.4f}")
    plt.show()

### Analiza wyników Zadania 1:
SSIM przyjmuje wartości od -1 do 1, gdzie 1 oznacza identyczne obrazy.
- Przy poziomie **0.1**, SSIM powinien być wysoki, a kształty ubrań/znaków wyraźne.
- Przy poziomie **0.3**, szczegóły zaczynają zanikać, a rekonstrukcja staje się 'rozmyta'.
- Przy poziomie **0.5**, błąd rekonstrukcji jest zazwyczaj na tyle duży, że autoenkoder gubi istotne cechy geometryczne, co objawia się znacznym spadkiem SSIM i trudnością w rozpoznaniu pierwotnego obiektu.

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

# 1. Progi do porównania
threshold_stat = threshold # To jest np.mean(train_loss) + np.std(train_loss)
threshold_visual = 0.05    # Przykładowy próg ustalony 'na oko' z histogramu

def get_predictions(model, data, threshold_val):
    reconstructions = model(data)
    loss = tf.keras.losses.mae(reconstructions, data)
    return tf.math.less(loss, threshold_val)

# Obliczamy predykcje dla obu progów
preds_stat = get_predictions(autoencoder, test_data, threshold_stat)
preds_visual = get_predictions(autoencoder, test_data, threshold_visual)

# 2. Tworzenie macierzy pomyłek
cm_stat = confusion_matrix(test_labels, preds_stat)
cm_visual = confusion_matrix(test_labels, preds_visual)

# 3. Wizualizacja
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

sns.heatmap(cm_stat, annot=True, fmt='d', ax=ax1, cmap='Blues')
ax1.set_title(f'Macierz pomyłek - Próg statystyczny ({threshold_stat:.4f})')
ax1.set_xlabel('Predykcja (1=Normalne, 0=Anomalia)')
ax1.set_ylabel('Rzeczywistość')

sns.heatmap(cm_visual, annot=True, fmt='d', ax=ax2, cmap='Greens')
ax2.set_title(f'Macierz pomyłek - Próg wizualny ({threshold_visual:.4f})')
ax2.set_xlabel('Predykcja (1=Normalne, 0=Anomalia)')
ax2.set_ylabel('Rzeczywistość')

plt.show()

### Analiza wyników Zadania 2:
Zauważ, jak zmiana progu wpływa na wyniki:
- **Niższy próg (bardziej restrykcyjny)**: Zwiększa liczbę wykrytych anomalii (Recall), ale może powodować, że zdrowe serca zostaną błędnie uznane za anomalie (False Positives).
- **Wyższy próg (bardziej liberalny)**: Zwiększa precyzję (Precision), ale model może przeoczyć subtelne anomalie (False Negatives).

Metoda wizualna pozwala 'nastroić' system pod konkretne wymagania (np. gdy zależy nam na niepominięciu żadnej choroby, nawet kosztem fałszywych alarmów).